<a href="https://colab.research.google.com/github/ochicken2104/Network-Automation-Victor-150871/blob/main/Lab_python%2CNetmiko_and_cisco_devnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install netmiko

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.9/262.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.4/642.4 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.9/223.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.0/161.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.2 MB/s eta 0:00:00


1. Defining Device Credentials

In [10]:
from netmiko import ConnectHandler # ConnectHandler - the primary function within
# Netmiko. Different vendors (Cisco, Juniper, Huawei, Arista, etc) use different
# command-line syntaxes, ConnectHandler acts as a universal adapter.
import getpass


# Device details for Cisco DevNet Sandbox

device = {
    'device_type': 'cisco_xe',
    'host': 'devnetsandboxiosxec8k.cisco.com',
    'username': 'victor.otieno',
    'password': 'ypIqc8NEHU-99_', #Public since we aint using getpass
    'port': 22,
}

print(f"Targeting: {device['host']}")

Targeting: devnetsandboxiosxec8k.cisco.com


2. The "Read" Operation(Retrieving Data)

In [11]:
try:
  # Establish SSH connection
  print("Connecting to the device ...")
  net_connect = ConnectHandler(**device)

  # Run a show command
  output = net_connect.send_command('show ip interface brief')

  print("\n--- Interface Status ---")
  print(output)

  # Disconnect
  net_connect.disconnect()

except Exception as e:
  print(f"Error connecting to the device: {e}")

Connecting to the device ...

--- Interface Status ---
Interface              IP-Address      OK? Method Status                Protocol
GigabitEthernet1       10.10.20.148    YES NVRAM  up                    up      
GigabitEthernet2       192.168.255.254 YES NVRAM  up                    up      
GigabitEthernet3       172.31.254.254  YES NVRAM  up                    up      
Loopback0              10.10.10.1      YES NVRAM  up                    up      
Loopback10             172.31.10.10    YES NVRAM  up                    up      
Loopback11             172.31.11.11    YES NVRAM  up                    up      
Loopback99             192.168.99.1    YES manual up                    up      
Loopback100            172.31.100.100  YES NVRAM  up                    up      
Loopback101            172.31.101.101  YES NVRAM  up                    up      
Loopback191            192.168.122.11  YES other  up                    up      
Loopback200            172.31.200.200  YES NVRAM  up  

3. The "Write" operation (Pushing confiigurations)

In [13]:
# Re-establish connections for the config change

# Redefine device with getpass for password to ensure correct authentication
device = {
    'device_type': 'cisco_xe',
    'host': 'devnetsandboxiosxec8k.cisco.com',
    'username': 'victor.otieno',
    'password': getpass.getpass("ypIqc8NEHU-99_"), # Using getpass for secure input
    'port': 22,
}

net_connect = ConnectHandler(**device)

config_commands = [
    'interface Loopback99',
    'description Configured via Python Netmiko',
    'ip address 192.168.99.1 255.255.255.255'
]

print("Sending configurations commands...")
output = net_connect.send_config_set(config_commands)
print(output)

# Verify the change
verification = net_connect.send_command("show ip interface brief | include Loopback99")
print("\nVerification")
print(verification)

net_connect.disconnect

ypIqc8NEHU-99_··········
Sending configurations commands...
configure terminal
Enter configuration commands, one per line.  End with CNTL/Z.
Router(config)#interface Loopback99
Router(config-if)#description Configured via Python Netmiko
Router(config-if)#ip address 192.168.99.1 255.255.255.255
Router(config-if)#end
Router#

Verification
Loopback99             192.168.99.1    YES manual up                    up      


<bound method BaseConnection.disconnect of <netmiko.cisco.cisco_ios.CiscoIosSSH object at 0x7a0e555a8770>>

4. Applying simple configuration such as loopback to switch


In [19]:
from netmiko import ConnectHandler
import getpass # Import getpass for the password prompt

# Device details
c8000v = {
    'device_type': 'cisco_xe', # Use cisco_xe if specifically targeting IOS-XE features
    'host': 'devnetsandboxiosxec8k.cisco.com',
    'username': 'victor.otieno',
    'password': getpass.getpass("ypIqc8NEHU-99_"),
    'port': 22,          # Default SSH port
}

# Establish SSH connection for this block
try:
  print("Connecting to the device ...")
  net_connect = ConnectHandler(**c8000v)

  # Configuration commands
  config_commands = [
      'interface Loopback0',
      'description Configured via Google Colab',
       'ip address 192.168.99.2 255.255.255.255'
  ]
  print("Sending configurations commands...")
  output = net_connect.send_config_set(config_commands)
  print(output)

  # Verify the change
  verification = net_connect.send_command("show ip interface brief | include0")
  print("\nVerification")
  print(verification)

  net_connect.disconnect()

except Exception as e:
  print(f"Error connecting to the device or sending commands: {e}")

ypIqc8NEHU-99_··········
Connecting to the device ...
Error connecting to the device or sending commands: TCP connection to device failed.

Common causes of this problem are:
1. Incorrect hostname or IP address.
2. Wrong TCP port.
3. Intermediate firewall blocking access.

Device settings: cisco_xe devnetsandboxiosxec8k.cisco.com:22


